# Rent model sandbox

Loads current `housing_listings` from Supabase and runs the **existing** training helpers:
- classic `train_and_evaluate_models` (OLS / RF / XGB / Durbin)
- `spatial_block_cv_xgb` (metrics)
- `compute_fair_rent_oof` (property-grouped OOF → `predictedrent`)
- CMA

Set `DB_URI` in `airflow/.env` (or your environment). This notebook does **not** write back to the DB unless you uncomment the update cell.

In [3]:
from pathlib import Path
import sys
import os

import pandas as pd
import numpy as np

# Resolve model package root whether you launch from repo root or notebooks/
NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATES = [
    NOTEBOOK_DIR / "model",
    NOTEBOOK_DIR.parent / "model",
    NOTEBOOK_DIR.parent.parent / "model",
    Path("/opt/airflow/model"),
]
MODEL_PATH = next((p for p in CANDIDATES if (p / "ml" / "model_training.py").exists()), None)
if MODEL_PATH is None:
    raise FileNotFoundError(f"Could not find airflow/model from {NOTEBOOK_DIR}")
MODEL_PATH = str(MODEL_PATH.resolve())
if MODEL_PATH not in sys.path:
    sys.path.insert(0, MODEL_PATH)
print("MODEL_PATH:", MODEL_PATH)

# Load DB_URI from airflow/.env if present
ENV_CANDIDATES = [
    Path(MODEL_PATH).parent / ".env",
    Path(MODEL_PATH) / ".env",
    NOTEBOOK_DIR / ".env",
]
try:
    from dotenv import load_dotenv
    for env_path in ENV_CANDIDATES:
        if env_path.exists():
            load_dotenv(env_path, override=False)
            print("Loaded env:", env_path)
            break
except ImportError:
    print("python-dotenv not installed; relying on existing environment vars")

assert os.getenv("DB_URI"), "Set DB_URI (e.g. in airflow/.env)"

MODEL_PATH: /Users/asm/Documents/GitHub/IthacaHousing/airflow/model
Loaded env: /Users/asm/Documents/GitHub/IthacaHousing/airflow/.env


In [5]:
import ml.model_training as model_training
import ml.data_preprocessing as data_preprocessing
import ml.comparative_market_analysis as comparative_market_analysis
import ml.rerun_rent_prediction as rerun_rent_prediction

ModuleNotFoundError: No module named 'matplotlib'

## 1) Load + prepare from Supabase
Uses `fetch_listings_for_retrain` / `prepare_features_from_db` (no scrape).

In [ ]:
raw = rerun_rent_prediction.fetch_listings_for_retrain()
apartments = rerun_rent_prediction.prepare_features_from_db(raw)

print(apartments.shape)
apartments[["ListingId", "PropertyId", "rent_per_person", "latitude", "longitude"]].head()

## 2) Build X / y

In [ ]:
X, y = model_training.define_X_Y_variables(apartments)
X, y = data_preprocessing.clean_up_x_y(X, y)
X = data_preprocessing.median_mode_imputation(X)
y_log = data_preprocessing.log_transform_prices(y)

X = X.reset_index(drop=True)
y_log = y_log.reset_index(drop=True) if hasattr(y_log, "reset_index") else pd.Series(y_log).reset_index(drop=True)
apartments = apartments.reset_index(drop=True)

print("Features:", list(X.columns))
print("n =", len(X), "| y (log) mean =", float(np.mean(y_log)))
X.head()

## 3) Classic model bake-off
`train_and_evaluate_models` → Linear / RF / XGB (spatial) + Spatial Durbin.

**Note:** these fit on the full sample (in-sample metrics), useful for relative comparison.

In [ ]:
apts_classic, results_classic, X_eval, X_spatial_eval, y_eval, best_name = (
    model_training.train_and_evaluate_models(X, y_log, apartments.copy())
)
print("Best:", best_name)
results_classic

## 4) Spatial block CV (XGBoost)
Honest holdout by spectral spatial clusters — same path as the production metrics step.

In [ ]:
X_coords = X.copy()
X_coords["latitude"] = apartments["latitude"].values
X_coords["longitude"] = apartments["longitude"].values

X_coords = model_training.add_h3_cells(X_coords, h3_reses=(6, 7))
X_coords = model_training.spatial_spectral_clustering(X_coords)

x_cols = [
    c for c in X_coords.columns
    if c not in ("spatial_cluster", "latitude", "longitude")
]

metrics_df, avg = model_training.spatial_block_cv_xgb(
    X_coords, y_log, x_cols, n_estimators=100, max_depth=6
)
display(metrics_df)
avg

## 5) Property-grouped OOF fair rent (+ CMA)
This is what production writes as `predictedrent`.

In [ ]:
apts_oof = apartments.copy()
apts_oof["spatial_cluster"] = X_coords["spatial_cluster"].values
for res in (6, 7):
    col = f"h3_{res}"
    if col in X_coords.columns:
        apts_oof[col] = X_coords[col].values

apts_oof = model_training.compute_fair_rent_oof(
    apartments_for_rent=apts_oof,
    y=y_log,
    x_cols=x_cols,
    h3_reses=(6, 7),
    k_knn=10,
    n_estimators=100,
    max_depth=6,
)

apts_oof["predictedrent"] = apts_oof["fair_rent"]
apts_oof["differenceinfairvalue"] = apts_oof["rent_per_person"] - apts_oof["predictedrent"]

X_cma = comparative_market_analysis.define_X_for_cma(apts_oof)
apts_oof = comparative_market_analysis.perform_cma(X_cma, apts_oof)

apts_oof[
    ["ListingId", "rent_per_person", "fair_rent", "predicted_rent_cma", "differenceinfairvalue"]
].describe()

In [ ]:
# Quick OOF error (log space vs $ space)
err = apts_oof["rent_per_person"] - apts_oof["fair_rent"]
{
    "mae_$": float(err.abs().mean()),
    "rmse_$": float(np.sqrt((err ** 2).mean())),
    "median_abs_$": float(err.abs().median()),
    "corr": float(apts_oof["rent_per_person"].corr(apts_oof["fair_rent"])),
}

## 6) Optional: write predictions back to Supabase
Uncomment to UPDATE `predictedrent` / CMA / `differenceinfairvalue` only.

In [ ]:
# rerun_rent_prediction.update_predictions_in_db(apts_oof)